<a href="https://colab.research.google.com/github/sparshap95-tech/NLP/blob/main/Minimal_Seq2Seq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# Dummy data:
input_seq = torch.tensor([[1, 2, 3]], dtype=torch.long)   # (batch, seq_len)
target_seq = torch.tensor([[3, 2, 1]], dtype=torch.long)

vocab_size = 10 # toy vocabulary size
embedding_dim = 8
hidden_size = 16


In [3]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)         # (batch, seq_len, embed_dim)
        outputs, hidden = self.rnn(embedded) # outputs: (batch, seq_len, hidden), hidden: (layers, batch, hidden)
        return hidden


In [4]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, target, hidden):
        embedded = self.embedding(target)      # (batch, seq_len, embed_dim)
        outputs, _ = self.rnn(embedded, hidden)# outputs: (batch, seq_len, hidden), hidden: (1, batch, hidden)
        logits = self.fc(outputs)              # (batch, seq_len, vocab_size)
        return logits


In [5]:
encoder = Encoder(vocab_size, embedding_dim, hidden_size)
decoder = Decoder(vocab_size, embedding_dim, hidden_size)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)

In [6]:
encoder_hidden = encoder(input_seq)  # hidden: (1, batch, hidden)
logits = decoder(target_seq, encoder_hidden)

In [7]:
logits.shape # (batch, seq_len, vocab_size)

torch.Size([1, 3, 10])

In [8]:
# One epoch training
encoder_hidden = encoder(input_seq)  # hidden: (1, batch, hidden)
logits = decoder(target_seq, encoder_hidden)

**During Inference**

In [9]:
def inference(input_seq):
    with torch.no_grad():
        hidden = encoder(input_seq)

        # Start with <SOS> token; here we use 0 as dummy <SOS>
        decoder_input = torch.tensor([[0]])   # (batch=1, seq_len=1)
        outputs = []

        for _ in range(input_seq.size(1)):
            logits = decoder(decoder_input, hidden)
            pred_token = logits.argmax(2)     # (batch, 1)
            outputs.append(pred_token.item())

            decoder_input = pred_token        # Next input is previous output

        return outputs

In [10]:
print("Predicted:", inference(input_seq))  # Should roughly be [3, 2, 1]


Predicted: [9, 3, 3]


**In case of Teacher Forcing**

In [11]:
class EncoderTF(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)

    def forward(self, x):
        embedded = self.embedding(x)         # (batch, seq_len, embed_dim)
        outputs, hidden = self.rnn(embedded) # outputs: (batch, seq_len, hidden), hidden: (1, batch, hidden)
        return hidden


In [12]:
class DecoderTF(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, target_seq, hidden, teacher_forcing_ratio=0.5):
        batch_size, target_len = target_seq.size()
        outputs = []

        input_token = target_seq[:, 0].unsqueeze(1)  # first input (<SOS> assumed)

        for t in range(1, target_len):
            embedded = self.embedding(input_token)               # (batch, 1, embed_dim)
            output, hidden = self.rnn(embedded, hidden)          # (batch, 1, hidden)
            logits = self.fc(output.squeeze(1))                  # (batch, vocab)
            outputs.append(logits.unsqueeze(1))                  # collect for all time steps

            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            next_input = target_seq[:, t] if teacher_force else logits.argmax(1)
            input_token = next_input.unsqueeze(1)

        return torch.cat(outputs, dim=1)  # (batch, target_len-1, vocab)


In [13]:
encoder2 = EncoderTF(vocab_size, embedding_dim, hidden_size)
decoder2 = DecoderTF(vocab_size, embedding_dim, hidden_size)

criterion2 = nn.CrossEntropyLoss()
optimizer2 = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=0.01)

In [14]:
# Forward pass
encoder_hidden2 = encoder2(input_seq)  # (1, batch, hidden)
logits2 = decoder2(target_seq, encoder_hidden2, teacher_forcing_ratio=0.7)

# Compute loss (note we shifted target by 1)
loss2 = criterion2(logits2.view(-1, vocab_size), target_seq[:, 1:].reshape(-1))
loss2.backward()
optimizer2.step()

print("Training loss with teacher forcing:", loss2.item())


Training loss with teacher forcing: 2.161985397338867


In [15]:
def inference_tf(input_seq, max_len=5):
    with torch.no_grad():
        hidden = encoder2(input_seq)
        input_token = torch.tensor([[0]])  # start with <SOS> (assumed 0)
        outputs = []

        for _ in range(max_len):
            embedded = decoder2.embedding(input_token)
            output, hidden = decoder2.rnn(embedded, hidden)
            logits = decoder2.fc(output.squeeze(1))
            pred_token = logits.argmax(1)
            outputs.append(pred_token.item())
            input_token = pred_token.unsqueeze(1)

        return outputs

print("Inference output:", inference_tf(input_seq))


Inference output: [3, 3, 3, 3, 3]
